# Feature A — Similar Product Recommendation

Goal: accept an ASIN or search term and return a deterministic ranked list with a short evidence-based reason. We compare TF-IDF, BM25, semantic embeddings, and a hybrid, then tune and select the final approach.


## 1. Setup and feature-specific preprocessing


In [ ]:
import sys, pickle, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

PROJECT = Path.cwd()
if not (PROJECT / 'analysis').exists(): PROJECT = PROJECT.parent
sys.path.insert(0, str(PROJECT))
from analysis.cleaning import load_clean
products, _ = load_clean()
df = products.drop_duplicates('asin').copy()
df = df[df['combined_text'].fillna('').str.strip().ne('')].sort_values('asin').reset_index(drop=True)
df['brand_norm'] = df['brand'].fillna('').str.lower().str.strip()
print('Products available:', len(df))
display(df[['asin','title','brand','search_keyword','combined_text']])


**Why these actions:** duplicate ASINs would repeat recommendations; empty text cannot be compared; sorting by ASIN and explicit score/ASIN tie-breaking guarantee identical results for identical input.


## 2. EDA and interpretation


In [ ]:
df['text_words'] = df['combined_text'].str.split().str.len()
fig, axes = plt.subplots(1,2,figsize=(14,5))
df['search_keyword'].value_counts().sort_values().plot.barh(ax=axes[0], title='Products per keyword')
df['text_words'].plot.hist(bins=30, ax=axes[1], title='Text length')
plt.tight_layout(); plt.show()
display(df[['text_words']].describe())
print('Interpretation: macro-average metrics by keyword so large groups do not dominate.')


## 3. Candidate models and tunable configurations


In [ ]:
corpus = df['combined_text'].tolist()
tfidf_configs = [
    {'ngram_range': (1,1), 'min_df': 1, 'sublinear_tf': False},
    {'ngram_range': (1,2), 'min_df': 1, 'sublinear_tf': True},
    {'ngram_range': (1,2), 'min_df': 2, 'sublinear_tf': True},
]
bm25_configs = [{'k1': k1, 'b': b} for k1 in (1.2,1.5,1.8) for b in (0.5,0.75,0.9)]

semantic_model_name = 'all-MiniLM-L6-v2'
semantic_model = SentenceTransformer(semantic_model_name)
semantic_embeddings = semantic_model.encode(corpus, normalize_embeddings=True, show_progress_bar=True)

tokenized = [re.findall(r'\b\w+\b', text.lower()) for text in corpus]
print('Candidates:', len(tfidf_configs), 'TF-IDF,', len(bm25_configs), 'BM25, 1 semantic, plus hybrids')


## 4. Correct deterministic ranking and evaluation


In [ ]:
def rank_scores(scores, query_index=None, top_k=5):
    candidates = [i for i in range(len(df)) if i != query_index]
    return sorted(candidates, key=lambda i: (-float(scores[i]), str(df.at[i,'asin'])))[:top_k]

def relevant(i, target_keyword):
    values = df.at[i, 'search_keywords'] if 'search_keywords' in df else None
    if isinstance(values, (list, tuple, set)):
        return target_keyword in values
    return df.at[i, 'search_keyword'] == target_keyword

def evaluate(score_function, sample_size=100, top_k=5):
    sample = df.sample(min(sample_size, len(df)), random_state=42).index
    precisions, reciprocal_ranks = [], []
    for idx in sample:
        ranked = rank_scores(score_function(idx), idx, top_k)
        matches = [relevant(i, df.at[idx,'search_keyword']) for i in ranked]
        precisions.append(sum(matches) / top_k)
        reciprocal_ranks.append(next((1/r for r,m in enumerate(matches,1) if m), 0))
    return {'precision_at_5': np.mean(precisions), 'mrr': np.mean(reciprocal_ranks)}


## 5. Fine-tuning and comparison


In [ ]:
results, fitted = [], {}
for config in tfidf_configs:
    vectorizer = TfidfVectorizer(stop_words='english', **config)
    matrix = vectorizer.fit_transform(corpus)
    name = f"tfidf_{config}"
    metrics = evaluate(lambda i, m=matrix: cosine_similarity(m[i], m).ravel())
    results.append({'model': name, **metrics}); fitted[name] = (vectorizer, matrix)

for config in bm25_configs:
    model = BM25Okapi(tokenized, **config)
    name = f"bm25_{config}"
    metrics = evaluate(lambda i, m=model: m.get_scores(tokenized[i]))
    results.append({'model': name, **metrics}); fitted[name] = model

semantic_scores = lambda i: semantic_embeddings @ semantic_embeddings[i]
results.append({'model': semantic_model_name, **evaluate(semantic_scores)})

best_tfidf_name = max((r for r in results if r['model'].startswith('tfidf')), key=lambda r:r['precision_at_5'])['model']
best_vec, best_tfidf = fitted[best_tfidf_name]
for weight in (0.5,0.7,0.85):
    def hybrid(i, w=weight):
        lexical = cosine_similarity(best_tfidf[i], best_tfidf).ravel()
        semantic = semantic_scores(i)
        return w * semantic + (1-w) * lexical
    results.append({'model': f'hybrid_semantic_{weight}', **evaluate(hybrid)})

results_df = pd.DataFrame(results).sort_values(['precision_at_5','mrr'], ascending=False)
display(results_df)
FINAL_MODEL = results_df.iloc[0]['model']
print('Selected:', FINAL_MODEL)


Category agreement is a reproducible proxy, not perfect relevance truth. Before submission, manually inspect a fixed set of queries and record whether the returned products are genuinely substitutable.


## 6. Final recommender: structured results and reasons


In [ ]:
def shared_terms(query_text, candidate_text, n=3):
    q = set(re.findall(r'\b[a-zA-Z]{3,}\b', query_text.lower()))
    c = set(re.findall(r'\b[a-zA-Z]{3,}\b', candidate_text.lower()))
    return sorted(q & c)[:n]

def final_scores(query_text, query_index=None):
    semantic_query = semantic_embeddings[query_index] if query_index is not None else semantic_model.encode([query_text], normalize_embeddings=True)[0]
    semantic = semantic_embeddings @ semantic_query
    lexical = cosine_similarity(best_vec.transform([query_text]), best_tfidf).ravel()
    return 0.7 * semantic + 0.3 * lexical

def recommend(query, top_k=5):
    matches = df.index[df['asin'].eq(str(query).upper())].tolist()
    query_index = matches[0] if matches else None
    query_text = df.at[query_index,'combined_text'] if query_index is not None else str(query)
    ranked = rank_scores(final_scores(query_text, query_index), query_index, top_k)
    rows = []
    target_brand = df.at[query_index,'brand_norm'] if query_index is not None else ''
    scores = final_scores(query_text, query_index)
    for i in ranked:
        reasons = []
        if target_brand and target_brand == df.at[i,'brand_norm']:
            reasons.append(f"same brand: {df.at[i,'brand']}")
        terms = shared_terms(query_text, df.at[i,'combined_text'])
        if terms: reasons.append('shared terms: ' + ', '.join(terms))
        if not reasons: reasons.append('high semantic and lexical similarity')
        rows.append({'asin': df.at[i,'asin'], 'title': df.at[i,'title'], 'score': round(float(scores[i]),4), 'reason': '; '.join(reasons)})
    return pd.DataFrame(rows)

display(recommend(df.iloc[0]['asin']))
display(recommend('noise cancelling headphones for sleeping'))


## 7. Save the selected model with product-order metadata


In [ ]:
models_dir = PROJECT / 'data' / 'models' / 'SimilarProductRecommendation'
models_dir.mkdir(parents=True, exist_ok=True)
bundle = {
    'asins': df['asin'].tolist(),
    'semantic_model_name': semantic_model_name,
    'semantic_embeddings': np.asarray(semantic_embeddings, dtype=np.float32),
    'tfidf_vectorizer': best_vec,
    'tfidf_matrix': best_tfidf,
    'hybrid_weights': {'semantic': 0.7, 'tfidf': 0.3},
    'evaluation': results_df.to_dict('records'),
}
with open(models_dir / 'recommendation_bundle.pkl', 'wb') as handle:
    pickle.dump(bundle, handle)
print('Saved:', models_dir / 'recommendation_bundle.pkl')
